In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, PatternedSequenceGenerator, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 
import os
import zipfile

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [3]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [4]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
      
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [5]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, total_lags=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')

        self.linear = nn.ModuleDict({
            str(ii): nn.Linear(hidden_size, vocab_size) 
            for ii in range(total_lags)
        })

        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)                      # [batch, seq_len, emb_dim]
        out, h = self.rnn(embedded, h)                    # out: [batch, seq_len, hidden_size]
        
        last_hidden = out[:, -1, :]       # [batch, memory_unit]
        
        output_heads = {ii: head(last_hidden) 
                        for ii, head in self.linear.items()}
        
        return output_heads, h

In [6]:
def multihead_loss(output_heads, targets, criterion=None, reduction="mean"):
    """
    output_heads: dict[str, Tensor], each [batch, vocab_size]
    targets: list[Tensor] or dict[str, Tensor], each [batch]
    criterion: loss function, defaults to CrossEntropyLoss
    reduction: 'mean' or 'sum'
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    losses = []
    for key, pred in output_heads.items():
        # if targets is a list, assume order matches keys
        if isinstance(targets, dict):
            target = targets[key]
        else:
            target = targets[int(key)]   # cast string key to int if using list
        # print(pred, target)
        losses.append(criterion(pred[0], target))

    if reduction == "mean":
        return sum(losses) / len(losses)
    elif reduction == "sum":
        return sum(losses)
    else:
        return losses   # return list if you want per-head loss


In [68]:
short_term_memory = 8
total_samples = 100000
working_memory = 1
hidden_size = 50
vocab_size = 8
embedding_dim = 10
lr = 1e-4


model = RNNEncoder(vocab_size, embedding_dim, hidden_size, total_lags=short_term_memory)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_random_sequence(total_samples, token_number=vocab_size)#get_sequence(total_samples, 2, 3, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
test_acc = []
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        y_pred, h = model(X)
    else:
        y_pred, h = model(X, hidden)

    loss = multihead_loss(y_pred, X[0])     
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        hidden = h.detach()
        total += 1

        key = str(short_term_memory-1)
        if X[0][-1] == y_pred[key].argmax():
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0
        
        if total%10000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {np.sum(correct)/total if total<1000 else np.sum(correct)/1000:.4f}')

    

Iter : 10001, loss: 1.4567, accuracy: 0.9840
Iter : 20001, loss: 0.8236, accuracy: 1.0000
Iter : 30001, loss: 0.7458, accuracy: 1.0000
Iter : 40001, loss: 0.4034, accuracy: 1.0000
Iter : 50001, loss: 0.5483, accuracy: 1.0000
Iter : 60001, loss: 0.2390, accuracy: 1.0000
Iter : 70001, loss: 0.3010, accuracy: 1.0000
Iter : 80001, loss: 0.0910, accuracy: 1.0000
Iter : 90001, loss: 0.1996, accuracy: 1.0000


In [98]:
h = None

with torch.no_grad():
    for ii in range(8):
        # num = np.random.randint(0,8)
        # if ii>9:
        #     print(num)
        y_pred, h = model(torch.tensor([[ii]]), h)

In [99]:
for ii in range(8):
    print(y_pred[str(ii)].argmax())

tensor(0)
tensor(1)
tensor(2)
tensor(3)
tensor(4)
tensor(5)
tensor(6)
tensor(7)


In [47]:
h

tensor([[[ 0.9841, -0.7512, -0.8083, -0.6386, -0.3582,  0.3423,  0.0164,
          -0.6483, -0.0958,  0.1929,  0.3589,  0.4909,  0.9034,  0.0492,
           0.2401, -0.2685,  0.7610,  0.5177, -0.8201, -0.6991,  0.0418,
          -0.3146,  0.4189,  0.2345,  0.1780, -0.5730,  0.6704,  0.2950,
           0.8354, -0.8538,  0.1247,  0.9116, -0.1866, -0.8037, -0.7859,
           0.3203,  0.6004, -0.5724,  0.9910, -0.7891]]])